# PaySim MoE Benchmark — Alpha Sweep

Federated fraud detection on PaySim dataset with comprehensive method comparison.

**Methods compared (11):**
- **FL (4)**: FedAvg · FedProx · FedNova · PersFL (MLP via cuml.accel)
- **ML experts (3)**: XGBoost (cuda) · LightGBM (cpu) · CatBoost (gpu)
- **MoE gates (4)**: Static · Performance · Confidence · Typology-Aware (oracle)

**Publishable-grade rigor:**
- **No label leakage** — dropped `nameOrig`, `nameDest`, `isFlaggedFraud`
- **Temporal split** — sort-by-step per bank, train=earliest / val=mid / test=latest
- **Per-model threshold tuning** on validation (maximize F2)
- **TypologyAwareGate** restricted to fraud transaction types only
- **Transparency columns** — `n_eval_banks`, `total_test_fraud`, per-method `threshold`

**Design matrix:** 1 dataset × 3 Dirichlet alphas {0.05, 0.1, 0.5} = 3 combos  
**Outputs:** `paysim_alpha{a}_benchmark.csv` × 3 + plots + learning curves  
**Features (10):** amt_log, amt_round, amt_thresh, type_enc, balance_diff_orig, balance_diff_dest, orig_zero_before, dest_zero_before, orig_zero_after, dest_zero_after

**Run environment:** Kaggle RAPIDS, GPU P100

In [ ]:
!pip install lightgbm xgboost imbalanced-learn catboost -q

In [ ]:
# ================================================================
# PAYSIM SINGLE-DATASET MoE BENCHMARK
# PaySim only, across Dirichlet alpha sweep {0.05, 0.1, 0.5}.
#
# FL  (4): FedAvg | FedProx | FedNova | PersFL  (sklearn MLP via cuml.accel)
# ML  (3): XGBoost (cuda) + LightGBM (cpu) + CatBoost (gpu)
# MoE (4): Static | Performance | Confidence | Typology-Aware (oracle)
#
# Rigor:
#  - No label leakage (nameOrig, nameDest, isFlaggedFraud dropped)
#  - Temporal split via 'step' column (train=early / val=mid / test=late)
#  - Per-model threshold tuned on validation (maximize F2)
#  - TypologyAwareGate routes only on fraud tx types (TRANSFER, CASH_OUT)
# ================================================================

from imblearn.over_sampling import SMOTE

# ── cuml / cupy: optional RAPIDS GPU acceleration ──
# Falls back to CPU sklearn + dummy flush if not in a RAPIDS environment
try:
    import cuml
    cuml.accel.install()
    import cupy as cp
    HAS_CUML = True
    print("cuml.accel installed — sklearn MLP will run on GPU")
except Exception:
    HAS_CUML = False
    print("cuml not available — using CPU sklearn MLP (no GPU acceleration)")

import os, gc, time, warnings, random
from collections import defaultdict

import numpy as np
import pandas as pd

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    average_precision_score, matthews_corrcoef,
    balanced_accuracy_score, confusion_matrix, fbeta_score
)

import xgboost as xgb
import lightgbm as lgb

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("WARNING: catboost not available — will skip CatBoost expert")

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

print("All imports OK")
print(f"  cuml GPU accel : {HAS_CUML}")
print(f"  CatBoost       : {HAS_CATBOOST}")

T0  = time.time()
OUT = '/kaggle/working' if os.path.exists('/kaggle') else '.'
def elapsed(): return f"{(time.time()-T0)/60:.1f}m"
def flush():
    gc.collect()
    if HAS_CUML:
        cp.get_default_memory_pool().free_all_blocks()

# ================================================================
# CONFIG
# ================================================================
N_BANKS    = 5          # PaySim simulates 5 banks
ALPHAS     = [0.05, 0.1, 0.5]

FL_STRATEGIES  = ['fedavg', 'fedprox', 'fednova', 'persfl']
FL_ROUNDS      = 20
LOCAL_EPOCHS   = 5
MLP_CAP        = 100_000
LR             = 0.001
FEDPROX_MU     = 0.01
TEST_FRAC      = 0.15
VAL_FRAC       = 0.15
THRESH_DEFAULT = 0.5
TUNE_THRESHOLD = True
THRESH_GRID    = np.arange(0.05, 0.95, 0.05)
AP_KS          = (50, 100, 200)

print("Config OK")

In [ ]:
# ================================================================
# PAYSIM DATA LOADING + PREPROCESSING
# ================================================================

def find_paysim():
    """Search common Kaggle paths for PaySim CSV."""
    candidates = [
        '/kaggle/input/paysim1/PS_20174392719_1491204439457_log.csv',
        '/kaggle/input/paysim-financial-crime-simulation/PS_20174392719_1491204439457_log.csv',
        '/kaggle/input/synthetic-financial-datasets-for-fraud-detection/PS_20174392719_1491204439457_log.csv',
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    # Fallback: walk input dirs
    for root, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.lower().endswith('.csv') and 'paysim' in f.lower():
                return os.path.join(root, f)
            if f.startswith('PS_') and f.endswith('.csv'):
                return os.path.join(root, f)
    return None


def load_paysim():
    path = find_paysim()
    if not path:
        raise FileNotFoundError(
            "PaySim CSV not found. Add dataset 'paysim1' to your Kaggle notebook."
        )
    print(f"Loading PaySim: {path}")
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    print(f"  {len(df):,} rows | fraud: {df['isFraud'].sum():,} "
          f"({df['isFraud'].mean()*100:.3f}%)")
    print(f"  Transaction types: {df['type'].value_counts().to_dict()}")
    return df


def preprocess_paysim(df):
    """
    Returns X, y, typ, t_col.

    Features (10 — leakage-free):
      amt_log          : log1p(amount)
      amt_round        : amount % 1000 == 0 (structuring flag)
      amt_thresh       : amount < 10000 (below reporting threshold)
      type_enc         : LabelEncoded transaction type (PAYMENT/TRANSFER/CASH_OUT etc.)
      balance_diff_orig: newbalanceOrig - oldbalanceOrg  (expected == -amount for legit)
      balance_diff_dest: newbalanceDest - oldbalanceDest (expected == +amount for legit)
      orig_zero_before : oldbalanceOrg == 0
      dest_zero_before : oldbalanceDest == 0
      orig_zero_after  : newbalanceOrig == 0
      dest_zero_after  : newbalanceDest == 0  (funds fully moved = laundering signal)

    Dropped:
      nameOrig, nameDest — account ID memorization / data leakage
      isFlaggedFraud     — derived from fraud label (leakage)
      step               — used only for temporal ordering (not in X)

    typ   : transaction type string — used for TypologyAwareGate routing
    t_col : step value — used for temporal train/val/test split ONLY
    """
    y     = df['isFraud'].astype(int).values
    t_col = df['step'].fillna(0).values.astype(np.float64)

    # typ: for fraud rows, use tx type; for legit, mark Normal
    # (TypologyAwareGate will skip Normal rows)
    typ = np.where(
        y == 1,
        ('PS_' + df['type'].astype(str)).values,
        'PS_Normal'
    )

    amt = df['amount'].fillna(0).abs()
    amt_log    = np.log1p(amt).values.astype(np.float32)
    amt_round  = (amt % 1000 == 0).astype(np.float32).values
    amt_thresh = (amt < 10_000).astype(np.float32).values

    le = LabelEncoder()
    type_enc = le.fit_transform(
        df['type'].astype(str).fillna('UNKNOWN')
    ).astype(np.float32)
    print(f"  Type encoder classes: {list(le.classes_)}")

    old_orig = df['oldbalanceOrg'].fillna(0).values.astype(np.float32)
    new_orig = df['newbalanceOrig'].fillna(0).values.astype(np.float32)
    old_dest = df['oldbalanceDest'].fillna(0).values.astype(np.float32)
    new_dest = df['newbalanceDest'].fillna(0).values.astype(np.float32)

    balance_diff_orig = (new_orig - old_orig).astype(np.float32)
    balance_diff_dest = (new_dest - old_dest).astype(np.float32)

    orig_zero_before = (old_orig == 0).astype(np.float32)
    dest_zero_before = (old_dest == 0).astype(np.float32)
    orig_zero_after  = (new_orig == 0).astype(np.float32)
    dest_zero_after  = (new_dest == 0).astype(np.float32)

    X = np.stack([
        amt_log, amt_round, amt_thresh,
        type_enc,
        balance_diff_orig, balance_diff_dest,
        orig_zero_before, dest_zero_before,
        orig_zero_after,  dest_zero_after,
    ], axis=1).astype(np.float32)

    print(f"  PaySim features: {X.shape}  (no nameOrig/nameDest/isFlaggedFraud; temporal split enabled)")
    print(f"  Fraud breakdown by type: "
          f"{pd.Series(df.loc[y==1,'type']).value_counts().to_dict()}")
    return X, y, typ, t_col


print("Data loading functions defined.")

In [ ]:
# ================================================================
# GENERIC DIRICHLET PARTITION + TEMPORAL SPLIT
# (same logic as original notebook, dataset-agnostic)
# ================================================================

def _allocate_fraud(n_fraud, n_banks, alpha):
    props = np.random.dirichlet([alpha] * n_banks)
    fcnts = (props * n_fraud).astype(int)
    diff = n_fraud - fcnts.sum()
    if diff > 0:
        for _ in range(diff):  fcnts[np.argmin(fcnts)] += 1
    elif diff < 0:
        for _ in range(-diff): fcnts[np.argmax(fcnts)] -= 1
    assert fcnts.sum() == n_fraud
    return fcnts


def partition_dataset(X, y, typ, t_col, n_banks, alpha, ds_name='PaySim'):
    np.random.seed(42)
    fraud_idx = np.where(y == 1)[0]
    legit_idx = np.where(y == 0)[0]
    np.random.shuffle(fraud_idx)
    np.random.shuffle(legit_idx)

    fcnts = _allocate_fraud(len(fraud_idx), n_banks, alpha)
    lper  = len(legit_idx) // n_banks

    banks = []
    fp = lp = 0
    for i in range(n_banks):
        ln  = lper if i < n_banks - 1 else len(legit_idx) - lp
        idx = np.concatenate([fraud_idx[fp:fp+fcnts[i]], legit_idx[lp:lp+ln]])
        banks.append(_split(i, X[idx], y[idx], typ[idx], t_col[idx],
                            f'{ds_name}-Bank{i}', ds_name.lower()))
        fp += fcnts[i]
        lp += ln

    print(f"\nPartition [PaySim] — {n_banks} banks | Dirichlet alpha={alpha} | temporal split:")
    for b in banks:
        print(f"  {b['name']:22s}: {b['n_total']:>10,} txns | "
              f"{b['n_fraud']:>6,} fraud ({b['fraud_frac']*100:.3f}%) | "
              f"train={b['n_train_fraud']} val={b['n_val_fraud']} test={b['n_test_fraud']} fraud")
    for b in banks:
        if b['n_test_fraud'] == 0:
            print(f"  NOTE: {b['name']} has 0 fraud in test — will report specificity/FPR instead")
    return banks


def _split(bid, X, y, typ, t, name, source):
    order = np.argsort(t, kind='stable')
    X, y, typ = X[order], y[order], typ[order]

    n = len(X)
    n_train = max(int(n * (1 - TEST_FRAC - VAL_FRAC)), 1)
    n_val   = max(int(n * VAL_FRAC), 1)
    if n_train + n_val >= n:
        n_train = max(n - 2, 1); n_val = 1

    Xtr, ytr, ttr = X[:n_train],              y[:n_train],              typ[:n_train]
    Xvl, yvl, tvl = X[n_train:n_train+n_val], y[n_train:n_train+n_val], typ[n_train:n_train+n_val]
    Xte, yte, tte = X[n_train+n_val:],        y[n_train+n_val:],        typ[n_train+n_val:]

    sc  = StandardScaler()
    Xtr = sc.fit_transform(Xtr).astype(np.float32)
    Xvl = sc.transform(Xvl).astype(np.float32)
    Xte = sc.transform(Xte).astype(np.float32)

    return dict(
        id=bid, name=name, source=source,
        X_train=Xtr, y_train=ytr, typ_train=ttr,
        X_val=Xvl,   y_val=yvl,   typ_val=tvl,
        X_test=Xte,  y_test=yte,  typ_test=tte,
        n_fraud=int(y.sum()), n_total=len(y),
        fraud_frac=float(y.mean()),
        n_train_fraud=int(ytr.sum()),
        n_val_fraud=int(yvl.sum()),
        n_test_fraud=int(yte.sum()),
    )


def safe_smote(X, y):
    if y.sum() < 5 or len(X) < 20: return X, y
    try:
        k = min(5, int(y.sum()) - 1)
        Xs, ys = SMOTE(random_state=42, k_neighbors=k).fit_resample(X, y)
        return Xs.astype(np.float32), ys.astype(np.int64)
    except Exception:
        return X, y


def get_mlp_data(bank):
    Xtr, ytr = bank['X_train'], bank['y_train']
    if len(Xtr) > MLP_CAP:
        fi = np.where(ytr == 1)[0]; li = np.where(ytr == 0)[0]
        nf = min(len(fi), MLP_CAP // 10); nl = min(len(li), MLP_CAP - nf)
        np.random.shuffle(fi); np.random.shuffle(li)
        idx = np.concatenate([fi[:nf], li[:nl]]); np.random.shuffle(idx)
        Xtr, ytr = Xtr[idx], ytr[idx]
    return safe_smote(Xtr, ytr)


print("Partition functions defined.")

In [ ]:
# ================================================================
# MLP + FL ALGORITHMS
# (FedAvg | FedProx | FedNova | PersFL — identical to original notebook)
# ================================================================

def make_mlp():
    return MLPClassifier(
        hidden_layer_sizes=(128, 64, 32), activation='relu',
        solver='adam', learning_rate_init=LR,
        max_iter=LOCAL_EPOCHS, random_state=42,
        warm_start=False, tol=1e-4)

def get_weights(m):
    return [c.copy() for c in m.coefs_] + [i.copy() for i in m.intercepts_]

def set_weights(m, w):
    n = len(m.coefs_)
    m.coefs_      = [w[i].copy() for i in range(n)]
    m.intercepts_ = [w[n+i].copy() for i in range(n)]

def avg_weights(wlist, counts=None):
    if counts is None: counts = [1] * len(wlist)
    total = sum(counts)
    return [sum(w[i] * c for w, c in zip(wlist, counts)) / total
            for i in range(len(wlist[0]))]

def mlp_proba(m, X):
    try:
        p = m.predict_proba(X)
        return p[:, 1] if p.ndim == 2 else p
    except Exception:
        return np.full(len(X), 0.5)

def init_mlp(X, y):
    m = make_mlp()
    m.fit(X[:min(len(X), 200)], y[:min(len(y), 200)])
    return m

def clone_mlp(ref, w):
    m = make_mlp()
    m.fit(ref.coefs_[0][:1] * 0, [0])   # initialise skeleton
    set_weights(m, w)
    return m

def tune_threshold(y_val, probs):
    if not TUNE_THRESHOLD or y_val.sum() == 0:
        return THRESH_DEFAULT
    best_t, best_f2 = THRESH_DEFAULT, -1
    for t in THRESH_GRID:
        preds = (probs >= t).astype(int)
        if preds.sum() == 0: continue
        f2 = fbeta_score(y_val, preds, beta=2, zero_division=0)
        if f2 > best_f2:
            best_f2, best_t = f2, t
    return best_t


# ── FL training loop ──────────────────────────────────────────

def run_fl(banks, strategy, n_features):
    print(f"  [{strategy.upper():7s}] ", end='', flush=True)

    # Seed global model with tiny bootstrap data
    all_X = np.vstack([b['X_train'][:50] for b in banks])
    all_y = np.concatenate([b['y_train'][:50] for b in banks])
    global_model = init_mlp(all_X, all_y)
    global_weights = get_weights(global_model)

    # PersFL: per-client personalized weights
    bank_weights = {b['id']: [w.copy() for w in global_weights] for b in banks}

    history = []

    # FedNova: gradient accumulators
    nova_tau = {b['id']: 1 for b in banks}

    for rnd in range(1, FL_ROUNDS + 1):
        local_ws, local_ns, local_deltas = [], [], []

        for b in banks:
            bid = b['id']
            Xtr, ytr = get_mlp_data(b)
            if len(Xtr) == 0: continue

            # Initialise local model from current global (or personal) weights
            # Skeleton needs both classes to avoid class mismatch error
            lm = make_mlp()
            fi = np.where(ytr == 1)[0]
            li = np.where(ytr == 0)[0]
            sk_idx = np.concatenate([fi[:1] if len(fi) > 0 else np.array([]),
                                     li[:1] if len(li) > 0 else np.array([])]).astype(int)
            if len(sk_idx) < 2:
                sk_idx = np.arange(min(10, len(Xtr)))
            lm.fit(Xtr[sk_idx], ytr[sk_idx])
            src_w = bank_weights[bid] if strategy == 'persfl' else global_weights
            set_weights(lm, src_w)
            
            if strategy == 'fedprox':
                w_before = get_weights(lm)
                lm.fit(Xtr, ytr)
                w_after = get_weights(lm)
                prox_w = [wa - FEDPROX_MU * (wa - wb)
                          for wa, wb in zip(w_after, w_before)]
                set_weights(lm, prox_w)
            else:
                lm.fit(Xtr, ytr)

            w_local = get_weights(lm)
            local_ws.append(w_local)
            local_ns.append(len(Xtr))

            if strategy == 'fednova':
                # delta = local - global; tau = local steps
                delta = [wl - wg for wl, wg in zip(w_local, global_weights)]
                local_deltas.append((delta, nova_tau[bid], len(Xtr)))

            if strategy == 'persfl':
                bank_weights[bid] = w_local

        if not local_ws:
            continue

        # ── Aggregation ──
        if strategy == 'fednova' and local_deltas:
            total_n = sum(n for _, _, n in local_deltas)
            # Normalized averaging of local updates
            agg_delta = [
                sum((d[i] / tau) * (n / total_n)
                    for d, tau, n in local_deltas)
                for i in range(len(local_deltas[0][0]))
            ]
            global_weights = [gw + ad for gw, ad in zip(global_weights, agg_delta)]
        else:
            # FedAvg aggregation (also used as base for FedProx, PersFL)
            global_weights = avg_weights(local_ws, local_ns)

        set_weights(global_model, global_weights)

        # ── Round metrics (avg AUPRC over all banks with val fraud) ──
        round_auprcs = []
        for b in banks:
            yt = b['y_val']
            if yt.sum() == 0: continue
            w_use = bank_weights[b['id']] if strategy == 'persfl' else global_weights
            lm_eval = make_mlp()
            lm_eval.fit(b['X_train'][:10], b['y_train'][:10])  # skeleton
            set_weights(lm_eval, w_use)
            probs = mlp_proba(lm_eval, b['X_val'])
            try:
                round_auprcs.append(average_precision_score(yt, probs))
            except Exception:
                pass
            history.append({'strategy': strategy, 'round': rnd,
                            'bank': b['name'], 'auprc': round_auprcs[-1] if round_auprcs else 0})

        avg_ap = np.mean(round_auprcs) if round_auprcs else 0
        if rnd % 5 == 0 or rnd == 1:
            print(f"r{rnd}(AUPRC={avg_ap:.3f}) ", end='', flush=True)

    print(f"done | {elapsed()}")
    return global_model, bank_weights, history


print("FL functions defined.")


In [ ]:
# ================================================================
# LOCAL ML EXPERTS: XGBoost | LightGBM | CatBoost
# ================================================================

class XGBExpert:
    name = 'xgb'
    def __init__(self, pw):
        # Use CUDA if GPU available, else fall back to CPU hist
        try:
            self.m = xgb.XGBClassifier(
                n_estimators=200, max_depth=6, learning_rate=0.05,
                scale_pos_weight=pw, tree_method='hist', device='cuda',
                eval_metric='aucpr', use_label_encoder=False,
                random_state=42, verbosity=0)
            self.m.fit([[0]*10], [0])  # probe GPU availability
        except Exception:
            self.m = xgb.XGBClassifier(
                n_estimators=200, max_depth=6, learning_rate=0.05,
                scale_pos_weight=pw, tree_method='hist', device='cpu',
                eval_metric='aucpr', use_label_encoder=False,
                random_state=42, verbosity=0)
        self.trained = False
    def fit(self, X, y):
        self.m.fit(X, y); self.trained = True
    def predict_proba(self, X):
        return self.m.predict_proba(X)[:, 1] if self.trained else np.full(len(X), 0.5)

class LGBMExpert:
    name = 'lgbm'
    def __init__(self, pw):
        self.m = lgb.LGBMClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.05,
            scale_pos_weight=pw, device='cpu',
            metric='average_precision', random_state=42, verbose=-1)
        self.trained = False
    def fit(self, X, y):
        self.m.fit(X, y); self.trained = True
    def predict_proba(self, X):
        return self.m.predict_proba(X)[:, 1] if self.trained else np.full(len(X), 0.5)

class CatBoostExpert:
    name = 'catboost'
    def __init__(self, pw):
        try:
            self.m = CatBoostClassifier(
                iterations=200, depth=6, scale_pos_weight=pw,
                task_type='GPU', devices='0', eval_metric='PRAUC',
                verbose=0, random_seed=42,
                subsample=0.8, bagging_temperature=0.5, bootstrap_type='Bernoulli')
        except Exception:
            self.m = CatBoostClassifier(
                iterations=200, depth=6, scale_pos_weight=pw,
                eval_metric='PRAUC', verbose=0, random_seed=42)
        self.trained = False
    def fit(self, X, y):
        try:
            self.m.fit(X, y)
        except Exception:
            pw = float((y==0).sum() / max((y==1).sum(), 1))
            self.m = CatBoostClassifier(
                iterations=200, depth=6, scale_pos_weight=pw,
                eval_metric='PRAUC', verbose=0, random_seed=42)
            self.m.fit(X, y)
        self.trained = True
    def predict_proba(self, X):
        return self.m.predict_proba(X)[:, 1] if self.trained else np.full(len(X), 0.5)


EXPERT_CLASSES = [XGBExpert, LGBMExpert]
if HAS_CATBOOST:
    EXPERT_CLASSES.append(CatBoostExpert)
EXPERT_NAMES = [c.name for c in EXPERT_CLASSES]


def train_experts(banks):
    experts = {}
    for b in banks:
        bid = b['id']; experts[bid] = {}
        Xtr, ytr = safe_smote(b['X_train'], b['y_train'])
        n0 = (ytr==0).sum(); n1 = max((ytr==1).sum(), 1); pw = float(n0/n1)
        if len(Xtr) == 0 or ytr.sum() == 0:
            for Cls in EXPERT_CLASSES:
                experts[bid][Cls.name] = None
            print(f"  {b['name']:22s} SKIPPED: no fraud in training")
            continue
        for Cls in EXPERT_CLASSES:
            try:
                m = Cls(pw); m.fit(Xtr, ytr)
                p = m.predict_proba(b['X_test'])
                pd_ = (p >= THRESH_DEFAULT).astype(int)
                if b['y_test'].sum() > 0:
                    f1 = f1_score(b['y_test'], pd_, zero_division=0)
                    ap = average_precision_score(b['y_test'], p)
                    print(f"  {b['name']:22s} {m.name.upper():8s}: F1={f1:.4f}  AUPRC={ap:.4f}")
                else:
                    tn = int(((b['y_test']==0)&(pd_==0)).sum())
                    fp = int(((b['y_test']==0)&(pd_==1)).sum())
                    print(f"  {b['name']:22s} {m.name.upper():8s}: (no test fraud) "
                          f"Spec={tn/max(tn+fp,1):.4f} FPR={fp/max(tn+fp,1):.4f}")
                experts[bid][m.name] = m
            except Exception as e:
                print(f"  {b['name']:22s} {Cls.name.upper():8s}: FAILED ({e})")
                experts[bid][Cls.name] = None
    return experts


print("Expert classes defined.")

In [ ]:
# ================================================================
# MoE GATES
# ================================================================

class StaticGate:
    name = 'moe_static'
    def get_weights(self, n, avail, **kw):
        w = np.where(avail, 1., 0.)
        return w / w.sum() if w.sum() > 0 else w
    def update(self, *a, **k): pass

class PerformanceGate:
    name = 'moe_performance'
    def __init__(self): self._w = {}
    def update(self, bid, vf1s, avail, **kw):
        s = np.where(avail, np.clip(vf1s, 0, None), 0.)
        self._w[bid] = s / s.sum() if s.sum() > 0 else np.ones(len(s)) / len(s)
    def get_weights(self, n, avail, bid=None, **kw):
        return self._w.get(bid, np.ones(n) / n)

class ConfidenceGate:
    name = 'moe_confidence'
    def __init__(self): self._w = {}
    def update(self, bid, val_probs_list, avail, **kw):
        confs = []
        for i, p in enumerate(val_probs_list):
            if avail[i] and p is not None and len(p) > 0:
                confs.append(float(np.mean(np.abs(p - 0.5) * 2)))
            else:
                confs.append(0.)
        s = np.array(confs)
        self._w[bid] = s / s.sum() if s.sum() > 0 else np.ones(len(s)) / len(s)
    def get_weights(self, n, avail, bid=None, **kw):
        return self._w.get(bid, np.ones(n) / n)

class TypologyAwareGate:
    """Oracle gate — routes by fraud typology. Upper bound; not deployable."""
    name = 'moe_typology_aware'
    def __init__(self):
        self._tbl = defaultdict(lambda: defaultdict(dict))
        self._fb  = {}
    def update(self, bid, typ_f1s, avail, val_f1s=None, **kw):
        for ei, (tf, av) in enumerate(zip(typ_f1s, avail)):
            if av: self._tbl[bid][ei] = tf
        if val_f1s is not None:
            s = np.where(avail, np.clip(val_f1s, 0, None), 0.)
            self._fb[bid] = s / s.sum() if s.sum() > 0 else np.ones(len(s)) / len(s)
    def get_weights(self, n, avail, bid=None, txn_typ=None, **kw):
        # PaySim-specific normal labels to skip
        NORMAL_LABELS = ('PS_Normal', 'Unknown')
        if txn_typ is None or bid not in self._tbl or txn_typ in NORMAL_LABELS:
            return self._fb.get(bid, np.ones(n) / n)
        w = np.zeros(n)
        for ei in range(n):
            if avail[ei] and ei in self._tbl[bid]:
                w[ei] = self._tbl[bid][ei].get(txn_typ, 0.)
        return w / w.sum() if w.sum() > 0 else self._fb.get(bid, np.ones(n) / n)


def _typ_f1(y_val, probs, typ_val, thresh):
    NORMAL_LABELS = ('PS_Normal', 'Unknown')
    preds = (probs >= thresh).astype(int)
    out = {}
    for t in np.unique(typ_val):
        if t in NORMAL_LABELS: continue
        mask = typ_val == t
        if mask.sum() < 2 or y_val[mask].sum() == 0: continue
        out[t] = f1_score(y_val[mask], preds[mask], zero_division=0)
    return out


print("MoE gates defined.")

In [ ]:
# ================================================================
# METRICS
# ================================================================

# PaySim typology weights
# Only TRANSFER and CASH_OUT are ever fraud in PaySim
TYP_W = {
    'PS_TRANSFER': 2.5,   # most common fraud type
    'PS_CASH_OUT': 2.5,   # second fraud type
}


def compute_metrics(y_true, probs, typ=None, thresh=THRESH_DEFAULT):
    preds  = (probs >= thresh).astype(int)
    n_pos  = int(y_true.sum())
    tn = int(((y_true==0)&(preds==0)).sum())
    fp = int(((y_true==0)&(preds==1)).sum())
    specificity = tn / max(tn + fp, 1)
    fpr         = fp / max(tn + fp, 1)

    if n_pos == 0:
        apk = {f'ap_at_{k}': 0. for k in AP_KS}
        return {'f1':0., 'precision':0., 'recall':0., 'auprc':0.,
                'mcc':0., 'f2':0., **apk,
                'typ_coverage':0., 'typ_wf1':0.,
                'specificity':float(specificity), 'fpr':float(fpr),
                'false_positives':fp, 'n_test_fraud':0, 'threshold':thresh}

    f1   = f1_score(y_true, preds, zero_division=0)
    prec = precision_score(y_true, preds, zero_division=0)
    rec  = recall_score(y_true, preds, zero_division=0)
    try:    auprc = average_precision_score(y_true, probs)
    except: auprc = float(y_true.mean())
    mcc  = matthews_corrcoef(y_true, preds) if preds.sum() > 0 else 0.
    b2   = 4; d = b2 * prec + rec
    f2   = (1 + b2) * prec * rec / d if d > 0 else 0.

    sidx = np.argsort(probs)[::-1]
    apk  = {f'ap_at_{k}': float(y_true[sidx[:min(k, len(y_true))]].sum() /
                                 max(min(k, len(y_true)), 1)) for k in AP_KS}

    typ_cov = 0.; typ_wf1 = 0.
    NORMAL_LABELS = ('PS_Normal', 'Unknown')
    if typ is not None:
        laund = [t for t in np.unique(typ) if t not in NORMAL_LABELS]
        if laund:
            typ_cov = sum(1 for t in laund
                if ((typ==t)&(y_true==1)&(preds==1)).sum() > 0) / len(laund)
        ws = wt = 0.
        for t in np.unique(typ):
            if t in NORMAL_LABELS: continue
            mask = typ == t
            if mask.sum() < 2 or y_true[mask].sum() == 0: continue
            f = f1_score(y_true[mask], preds[mask], zero_division=0)
            w = TYP_W.get(t, 1.5)
            ws += w * f; wt += w
        typ_wf1 = ws / max(wt, 1e-8)

    return {'f1':f1, 'precision':prec, 'recall':rec, 'auprc':auprc,
            'mcc':mcc, 'f2':f2, **apk,
            'typ_coverage':typ_cov, 'typ_wf1':typ_wf1,
            'specificity':float(specificity), 'fpr':float(fpr),
            'false_positives':fp, 'n_test_fraud':n_pos, 'threshold':thresh}


def agg(ml):
    if not ml: return {}
    return {k: round(float(np.mean([m[k] for m in ml])), 4) for k in ml[0]}

def fairness(f1s, lf=None):
    r = dict(client_equity=round(float(np.std(f1s)), 4),
             worst_bank_f1=round(float(min(f1s)), 4),
             best_bank_f1=round(float(max(f1s)), 4))
    if lf and len(lf) == len(f1s):
        r['collab_gain'] = round(float(np.mean([a-b for a,b in zip(f1s,lf)])), 4)
    return r


print("Metrics functions defined.")

In [ ]:
# ================================================================
# EVALUATION + PLOTS
# ================================================================

def evaluate_all(banks, fl_results, experts):
    rows  = []
    n_fl  = len(fl_results)
    n_ml  = len(EXPERT_NAMES)
    n_exp = n_fl + n_ml

    # Local-only F1 baseline per bank (for collab_gain)
    loc_f1 = {}
    for b in banks:
        bid = b['id']; yt = b['y_test']; best = 0.
        for en in EXPERT_NAMES:
            m = experts[bid].get(en)
            if m and m.trained and yt.sum() > 0:
                p = m.predict_proba(b['X_test'])
                t = tune_threshold(b['y_val'], m.predict_proba(b['X_val']))
                best = max(best, f1_score(yt, (p >= t).astype(int), zero_division=0))
        loc_f1[bid] = best

    def _row(strat, mtype, bm, lf=None, n_eval=None, n_with_fraud=None, total_fraud=None):
        a  = agg(bm)
        fa = fairness([m['f1'] for m in bm], lf)
        extra = {}
        if n_eval is not None:       extra['n_eval_banks']       = n_eval
        if n_with_fraud is not None: extra['n_banks_with_fraud'] = n_with_fraud
        if total_fraud is not None:  extra['total_test_fraud']   = total_fraud
        return {'strategy': strat, 'model_type': mtype, **a, **fa, **extra}

    total_fraud = sum(int(b['y_test'].sum()) for b in banks)

    # ── FL models ──
    for strat, (fl_model, bw, _) in fl_results.items():
        bm = []; lf = []
        for b in banks:
            yt = b['y_test']; tte = b.get('typ_test')
            pv = mlp_proba(bw[b['id']], b['X_val']) if strat=='persfl' and b['id'] in bw else mlp_proba(fl_model, b['X_val'])
            pt = mlp_proba(bw[b['id']], b['X_test']) if strat=='persfl' and b['id'] in bw else mlp_proba(fl_model, b['X_test'])
            t_tune = tune_threshold(b['y_val'], pv)
            bm.append(compute_metrics(yt, pt, tte, thresh=t_tune))
            lf.append(loc_f1.get(b['id'], 0))
        n_with_fraud = sum(1 for b in banks if b['y_test'].sum() > 0)
        rows.append(_row(strat, 'fl', bm, lf, len(banks), n_with_fraud, total_fraud))

    # ── Local experts ──
    for en in EXPERT_NAMES:
        bm = []
        for b in banks:
            yt = b['y_test']; tte = b.get('typ_test')
            m  = experts[b['id']].get(en)
            if m and m.trained:
                pv = m.predict_proba(b['X_val'])
                pt = m.predict_proba(b['X_test'])
                t_tune = tune_threshold(b['y_val'], pv)
            else:
                pt = np.full(len(yt), 0.5); t_tune = THRESH_DEFAULT
            bm.append(compute_metrics(yt, pt, tte, thresh=t_tune))
        n_with_fraud = sum(1 for b in banks if b['y_test'].sum() > 0)
        rows.append(_row(en, 'local_expert', bm, None, len(banks), n_with_fraud, total_fraud))

    # ── MoE Gates ──
    gates = [StaticGate(), PerformanceGate(), ConfidenceGate(), TypologyAwareGate()]
    for gate in gates:
        bm = []; lf = []
        for b in banks:
            bid = b['id']; yt = b['y_test']
            tte = b.get('typ_test', np.array(['PS_Normal']*len(yt)))
            tvl = b.get('typ_val',  np.array(['PS_Normal']*len(b['y_val'])))

            vps=[]; tps=[]; vf1s=[]; avl=[]; tyf=[]

            for strat, (fl_model, bw, _) in fl_results.items():
                pv = mlp_proba(bw[bid], b['X_val']) if strat=='persfl' and bid in bw else mlp_proba(fl_model, b['X_val'])
                pt_probs = mlp_proba(bw[bid], b['X_test']) if strat=='persfl' and bid in bw else mlp_proba(fl_model, b['X_test'])
                t_tune = tune_threshold(b['y_val'], pv)
                vps.append(pv); tps.append(pt_probs)
                vf1s.append(fbeta_score(b['y_val'],(pv>=t_tune).astype(int),beta=2,zero_division=0) if b['y_val'].sum()>0 else 0.)
                tyf.append(_typ_f1(b['y_val'],pv,tvl,t_tune) if b['y_val'].sum()>0 else {})
                avl.append(True)

            for en in EXPERT_NAMES:
                m = experts[bid].get(en)
                if m and m.trained:
                    pv = m.predict_proba(b['X_val'])
                    pt_probs = m.predict_proba(b['X_test'])
                    t_tune = tune_threshold(b['y_val'], pv)
                    vps.append(pv); tps.append(pt_probs)
                    vf1s.append(fbeta_score(b['y_val'],(pv>=t_tune).astype(int),beta=2,zero_division=0) if b['y_val'].sum()>0 else 0.)
                    tyf.append(_typ_f1(b['y_val'],pv,tvl,t_tune) if b['y_val'].sum()>0 else {})
                    avl.append(True)
                else:
                    vps.append(np.full(len(b['y_val']),0.5)); tps.append(np.full(len(yt),0.5))
                    vf1s.append(0.); tyf.append({}); avl.append(False)

            avl=np.array(avl); vf1s=np.array(vf1s)

            if isinstance(gate, TypologyAwareGate):
                gate.update(bid, tyf, avl, val_f1s=vf1s)
                ap = np.zeros(len(yt))
                for ut in np.unique(tte):
                    mask = tte==ut
                    w = gate.get_weights(n_exp,avl,bid=bid,txn_typ=ut)
                    for i,pt_probs in enumerate(tps): ap[mask] += w[i]*pt_probs[mask]
            elif isinstance(gate, PerformanceGate):
                gate.update(bid,vf1s,avl)
                w=gate.get_weights(n_exp,avl,bid=bid)
                ap=sum(w[i]*tps[i] for i in range(n_exp))
            elif isinstance(gate, ConfidenceGate):
                gate.update(bid,vps,avl)
                w=gate.get_weights(n_exp,avl,bid=bid)
                ap=sum(w[i]*tps[i] for i in range(n_exp))
            else:
                w=gate.get_weights(n_exp,avl)
                ap=sum(w[i]*tps[i] for i in range(n_exp))

            # MoE val probs for threshold tuning
            if isinstance(gate, TypologyAwareGate):
                ap_val=np.zeros(len(b['y_val']))
                for ut in np.unique(tvl):
                    mask=tvl==ut
                    w_val=gate.get_weights(n_exp,avl,bid=bid,txn_typ=ut)
                    for i,pv_p in enumerate(vps): ap_val[mask]+=w_val[i]*pv_p[mask]
            else:
                if isinstance(gate,(PerformanceGate,ConfidenceGate)):
                    w_val=gate.get_weights(n_exp,avl,bid=bid)
                else:
                    w_val=gate.get_weights(n_exp,avl)
                ap_val=sum(w_val[i]*vps[i] for i in range(n_exp))

            t_gate=tune_threshold(b['y_val'],ap_val)
            bm.append(compute_metrics(yt,ap,tte,thresh=t_gate))
            lf.append(loc_f1.get(bid,0))

        n_with_fraud=sum(1 for b in banks if b['y_test'].sum()>0)
        rows.append(_row(gate.name,'moe',bm,lf,len(banks),n_with_fraud,total_fraud))
    return rows


# ── Plotting ──────────────────────────────────────────────────

COLORS = {
    'fedavg':'#4F46A0', 'fedprox':'#0F6E56', 'fednova':'#854F0B', 'persfl':'#185FA5',
    'xgb':'#A32D2D', 'lgbm':'#D85A30', 'catboost':'#F2A154',
    'moe_static':'#AAAAAA', 'moe_performance':'#1D9E75',
    'moe_confidence':'#E04B8A', 'moe_typology_aware':'#9333EA',
}


def plot_results(df_bm, fl_hist, tag):
    strats = sorted(df_bm['strategy'].unique())
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle(
        f'{tag} — Temporal Split, Dirichlet | PaySim\n'
        'MLP FL (4) + XGB/LGBM/CatBoost + MoE (4 gates) [leakage-free + tuned thresholds]',
        fontsize=10, fontweight='bold')

    ax = axes[0]; x = np.arange(len(strats)); w = 0.22
    for mi, (m, lbl, c) in enumerate([('auprc','AUPRC','#2E4057'),
                                       ('f2','F2','#048A81'),
                                       ('mcc','MCC','#54C6EB')]):
        vals = [float(df_bm[df_bm['strategy']==s][m].mean()) for s in strats]
        ax.bar(x+(mi-1)*w, vals, w, label=lbl, color=c, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace('moe_','MoE-') for s in strats],
                       fontsize=7, rotation=45, ha='right')
    ax.set_ylim(0,1); ax.legend(fontsize=8, frameon=False)
    ax.set_title('Primary Metrics (avg across ALL banks)', fontsize=9, fontweight='bold')
    ax.spines[['top','right']].set_visible(False); ax.grid(axis='y', alpha=0.25)

    ax = axes[1]
    for si, s in enumerate(strats):
        c = COLORS.get(s, '#999')
        ap = float(df_bm[df_bm['strategy']==s]['auprc'].mean())
        rc = float(df_bm[df_bm['strategy']==s]['recall'].mean())
        ax.bar(si-0.18, ap, 0.35, color=c, alpha=0.85)
        ax.bar(si+0.18, rc, 0.35, color=c, alpha=0.4, edgecolor=c, linewidth=1.5)
    ax.set_xticks(range(len(strats)))
    ax.set_xticklabels([s.replace('moe_','MoE-') for s in strats],
                       fontsize=7, rotation=45, ha='right')
    ax.set_ylim(0,1)
    ax.set_title('AUPRC (solid) vs Recall (hollow)', fontsize=9, fontweight='bold')
    ax.spines[['top','right']].set_visible(False); ax.grid(axis='y', alpha=0.25)

    ax = axes[2]
    dfh = pd.DataFrame(fl_hist)
    if not dfh.empty:
        for strat in ['fedavg','fedprox','fednova','persfl']:
            sub = dfh[dfh['strategy']==strat]
            if sub.empty: continue
            g = sub.groupby('round')['auprc'].mean().reset_index()
            ax.plot(g['round'], g['auprc'], label=strat.upper(),
                    color=COLORS.get(strat,'#999'), lw=2, marker='o', ms=4)
    ax.set_xlabel('FL Round'); ax.set_ylim(0,1)
    ax.set_title('FL Learning Curves (avg AUPRC)', fontsize=9, fontweight='bold')
    ax.legend(fontsize=8, frameon=False)
    ax.spines[['top','right']].set_visible(False); ax.grid(alpha=0.25)

    plt.tight_layout()
    p = f'{OUT}/{tag.lower()}_benchmark_results.png'
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"Saved: {p}")


def print_summary(df_bm, label):
    print(f"\n{'='*70}")
    print(f"{label} BENCHMARK — Temporal Split, Dirichlet ({N_BANKS} Banks)")
    print("All banks included. Fraud metrics=0 for banks with no test fraud.")
    print("="*70)
    cols = ['strategy','auprc','mcc','f2','f1','recall','specificity','fpr',
            'ap_at_100','typ_coverage','typ_wf1',
            'client_equity','worst_bank_f1','collab_gain',
            'threshold','n_eval_banks','n_banks_with_fraud','total_test_fraud']
    avail = [c for c in cols if c in df_bm.columns]
    print(df_bm[avail].sort_values('auprc', ascending=False).to_string(index=False))
    best = df_bm.loc[df_bm['auprc'].idxmax()]
    print(f"\nBEST (AUPRC): {best['strategy']:25s}  "
          f"AUPRC={best['auprc']:.4f}  MCC={best['mcc']:.4f}  F2={best['f2']:.4f}")


print("Evaluation + plot functions defined.")

In [ ]:
# ================================================================
# MAIN: PaySim × Alpha Sweep
# 1 dataset × 3 alphas = 3 benchmark combos
# Each combo: 4 FL + 3 ML experts + 4 MoE gates = 11 methods
# ================================================================

all_results = {}

# ── Load & Preprocess ONCE (expensive) ──
print("#" * 60)
print("DATASET: PaySim")
print("#" * 60)

df = load_paysim()
X, y, typ, t_col = preprocess_paysim(df)
del df; flush()
print(f"Preprocessing done | {elapsed()}")

# ── Alpha Sweep ──
for alpha in ALPHAS:
    tag = f"paysim_alpha{alpha}"
    print(f"\n{'='*60}")
    print(f"  PaySim | alpha={alpha} | {N_BANKS} banks")
    print(f"{'='*60}")

    # Partition (Dirichlet on fraud, even legit, temporal split inside each bank)
    banks = partition_dataset(X, y, typ, t_col, N_BANKS, alpha, 'PaySim')

    # FL Training (all 4 strategies)
    fl_results = {}; all_hist = []
    print(f"\nFL training:")
    for strat in FL_STRATEGIES:
        fl_model, bw, hist = run_fl(banks, strat, X.shape[1])
        if strat == 'persfl':
            init_X = np.vstack([b['X_train'][:50] for b in banks])
            init_y = np.concatenate([b['y_train'][:50] for b in banks])
            tm  = init_mlp(init_X, init_y)
            bw_m = {bid: clone_mlp(tm, w) for bid, w in bw.items()}
            fl_results[strat] = (fl_model, bw_m, hist)
        else:
            fl_results[strat] = (fl_model, {}, hist)
        for h in hist: h['alpha'] = alpha
        all_hist.extend(hist)
        flush()

    # Local Experts
    print(f"\nLocal experts (XGB cuda + LGBM cpu"
          f"{' + CatBoost gpu' if HAS_CATBOOST else ''})...")
    experts = train_experts(banks)
    flush()

    # Evaluate + MoE
    print(f"\nEvaluating all models + MoE gates...")
    bm_rows = evaluate_all(banks, fl_results, experts)
    for r in bm_rows:
        r['dataset'] = 'PaySim'
        r['alpha']   = alpha

    # Save outputs
    df_bm = pd.DataFrame(bm_rows)
    df_bm.to_csv(f'{OUT}/{tag}_benchmark.csv', index=False)
    pd.DataFrame(all_hist).to_csv(f'{OUT}/{tag}_fl_history.csv', index=False)

    print_summary(df_bm, f"PaySim (alpha={alpha})")
    plot_results(df_bm, all_hist, f"PaySim_alpha{alpha}")

    all_results[tag] = df_bm

    del banks, fl_results, experts, all_hist
    flush()
    print(f"\nPaySim alpha={alpha} done | {elapsed()}")

del X, y, typ, t_col
flush()

# ── Consolidated Results (3 alpha combos) ──
if all_results:
    combined = pd.concat(all_results.values(), ignore_index=True)
    combined.to_csv(f'{OUT}/paysim_all_alphas_combined.csv', index=False)

    print(f"\n{'='*70}")
    print("CONSOLIDATED — PaySim × ALL ALPHAS")
    print(f"{'='*70}")

    piv = combined.pivot_table(
        index='alpha', columns='strategy', values='auprc', aggfunc='mean'
    ).round(4)
    print("\nAUPRC pivot (alpha × strategy):")
    print(piv.to_string())

    piv_mcc = combined.pivot_table(
        index='alpha', columns='strategy', values='mcc', aggfunc='mean'
    ).round(4)
    print("\nMCC pivot (alpha × strategy):")
    print(piv_mcc.to_string())

print(f"\n{'='*60}")
print("ALL RUNS COMPLETE")
print(f"{'='*60}")
print(f"Total runtime: {elapsed()}")
print("\nOutputs:")
for f in sorted(os.listdir(OUT)):
    if f.startswith('paysim') and f.endswith(('.csv', '.png')):
        print(f"  {f}")